# 02. EDA (10/26) — 최지훈 A 업무

`01_data_prep.ipynb`로 만든 `processed/` 파일을 읽어서 PPT용 그래프 5개와 요약표를 만든다.

| 그래프 | 보여 줄 것 |
|---|---|
| fig1 평점 분포 | 5점 쏠림 → 동점 처리, Top-N z-점수의 근거 |
| fig2 사용자당 평점 수 | 대부분 10개 미만 → 5-core를 그대로 쓴 이유 |
| fig3 train 평점 구간 | 3~4개 구간이 약 절반 → 콜드스타트 비교 표본 충분 |
| fig4 상품 인기도 롱테일 | 소수 상품 집중 → 인기도 기준 모델·프로토콜 ①②③ |
| fig5 연도 분포 | test가 train보다 3~4년 뒤 → 결과 해석 시 주의 |

그래프 글자는 영어로 둔다 (Colab 기본 폰트에 한글이 없어서 깨짐). 그림은 Drive `results/eda/`에 저장된다.

In [ ]:
# ===== 0. 설정 =====
import os, sys, subprocess, json, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/qhfl3644/team10-recsys.git', REPO], check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
except ImportError:                       # Colab 밖에서 테스트할 때
    REPO = os.environ.get('TEAM10_REPO', '..')
sys.path.insert(0, REPO)
from config import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = f'{RESULT_DIR}/eda'
os.makedirs(FIG_DIR, exist_ok=True)

COLOR = {'Video_Games': '#2a78d6', 'Toys_and_Games': '#eb6834'}     # 데이터셋 색은 모든 그림에서 고정
LABEL = {'Video_Games': 'Video Games (build)', 'Toys_and_Games': 'Toys & Games (test)'}
SPLIT_COLOR = {'train': '#6250d6', 'test': '#1baf7a'}
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': INK, 'font.size': 10,
    'legend.frameon': False,
})

def save(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('saved:', f'{FIG_DIR}/{name}.png')

In [ ]:
# ===== 1. 데이터 읽기 =====
data = {}
for ds in DATASETS:
    data[ds] = {s: pd.read_parquet(proc_path(ds, s)) for s in ['ratings_all', 'train', 'valid', 'test']}
    print(ds, {s: f'{len(df):,}' for s, df in data[ds].items()})

In [ ]:
# ===== 2. 요약 통계 (제안서 2.2절 수치와 대조) =====
PROPOSAL = {   # 제안서 2.2절
    'Video_Games':    {'5점 비율(%)': 63.8, '평균 평점': 4.27, '사용자당 평점 평균': 8.6, '사용자당 평점 중앙값': 6,
                       '10개 미만 사용자(%)': 77.8, '정확히 5개 사용자(%)': 32.1,
                       'train 3~4개 사용자': 48_624, 'train 5~9개 사용자': 32_212, 'train 10개+ 사용자': 13_926,
                       'train 연도 중앙값': 2015, 'test 연도 중앙값': 2019},
    'Toys_and_Games': {'5점 비율(%)': 73.0, '평균 평점': 4.46, '사용자당 평점 평균': 8.9, '사용자당 평점 중앙값': 7,
                       '10개 미만 사용자(%)': 76.0, '정확히 5개 사용자(%)': 30.4,
                       'train 3~4개 사용자': 212_587, 'train 5~9개 사용자': 149_818, 'train 10개+ 사용자': 69_859,
                       'train 연도 중앙값': 2018, 'test 연도 중앙값': 2021},
}

def year_of(ts):
    return pd.to_datetime(ts, unit='ms').dt.year

def segment(n):
    return pd.cut(n, bins=[2, 4, 9, np.inf], labels=['3-4', '5-9', '10+'])

stats, cache = {}, {}
for ds in DATASETS:
    d = data[ds]
    allr, tr, te = d['ratings_all'], d['train'], d['test']
    per_user = allr.groupby('user_id').size()
    tr_user = tr.groupby('user_id').size()
    seg = segment(tr_user).value_counts().reindex(['3-4', '5-9', '10+'])
    n_u, n_i = allr.user_id.nunique(), allr.item_id.nunique()
    stats[ds] = {
        '평점 수': len(allr), '사용자 수': n_u, '상품 수': n_i,
        '밀도(%)': round(len(allr) / (n_u * n_i) * 100, 3),
        '5점 비율(%)': round((allr.rating == 5).mean() * 100, 1),
        '평균 평점': round(allr.rating.mean(), 2),
        '사용자당 평점 평균': round(per_user.mean(), 1),
        '사용자당 평점 중앙값': int(per_user.median()),
        '10개 미만 사용자(%)': round((per_user < 10).mean() * 100, 1),
        '정확히 5개 사용자(%)': round((per_user == 5).mean() * 100, 1),
        'train 3~4개 사용자': int(seg['3-4']), 'train 5~9개 사용자': int(seg['5-9']), 'train 10개+ 사용자': int(seg['10+']),
        'train 연도 중앙값': int(year_of(tr.timestamp).median()),
        'test 연도 중앙값': int(year_of(te.timestamp).median()),
    }
    cache[ds] = {'per_user': per_user, 'tr_user': tr_user, 'seg': seg}

rows = []
for ds in DATASETS:
    for k, v in stats[ds].items():
        p = PROPOSAL[ds].get(k)
        rows.append({'데이터셋': ds, '항목': k, '우리 값': v, '제안서': p if p is not None else '',
                     '일치': '' if p is None else ('O' if v == p else '확인')})
summary = pd.DataFrame(rows)
summary.to_csv(f'{FIG_DIR}/eda_summary.csv', index=False, encoding='utf-8-sig')
summary

In [ ]:
# ===== fig1. 평점 분포 =====
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(1, 6)
w = 0.38
for j, ds in enumerate(DATASETS):
    share = data[ds]['ratings_all'].rating.value_counts(normalize=True).reindex(x, fill_value=0) * 100
    bars = ax.bar(x + (j - 0.5) * (w + 0.02), share.values, width=w, color=COLOR[ds], label=LABEL[ds])
    ax.annotate(f'{share[5]:.1f}%', (bars[-1].get_x() + w / 2, share[5]), xytext=(0, 3),
                textcoords='offset points', ha='center', fontsize=9, color=INK)
ax.set_xticks(x, [f'{i}★' for i in x])
ax.set_ylabel('Share of ratings (%)')
ax.set_title('Ratings are concentrated at 5 stars')
ax.legend(loc='upper left')
save(fig, 'fig1_rating_distribution')

In [ ]:
# ===== fig2. 사용자당 평점 수 =====
CAP = 30
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, ds in zip(axes, DATASETS):
    pu = cache[ds]['per_user']
    clipped = pu.clip(upper=CAP)
    share = clipped.value_counts(normalize=True).sort_index() * 100
    ax.bar(share.index, share.values, width=0.8, color=COLOR[ds])
    med = pu.median()
    ax.axvline(med, color=MUTED, linestyle='--', linewidth=1)
    ax.text(med + 0.5, share.max() * 0.92, f'median {med:.0f}', color=MUTED, fontsize=9)
    ax.text(CAP, share.loc[CAP], f'{CAP}+', ha='center', va='bottom', fontsize=8, color=MUTED)
    ax.set_title(f'{LABEL[ds]}: {(pu < 10).mean()*100:.0f}% of users have < 10 ratings', fontsize=11)
    ax.set_xlabel('Ratings per user (5-core, all splits)')
axes[0].set_ylabel('Share of users (%)')
save(fig, 'fig2_ratings_per_user')

In [ ]:
# ===== fig3. train 평점 구간 (콜드스타트 구간) =====
fig, ax = plt.subplots(figsize=(7, 4))
segs = ['3-4', '5-9', '10+']
x = np.arange(len(segs))
w = 0.38
for j, ds in enumerate(DATASETS):
    seg = cache[ds]['seg']
    share = seg / seg.sum() * 100
    bars = ax.bar(x + (j - 0.5) * (w + 0.02), share.values, width=w, color=COLOR[ds], label=LABEL[ds])
    for b, s, n in zip(bars, share.values, seg.values):
        ax.annotate(f'{s:.0f}%\n({n:,})', (b.get_x() + w / 2, s), xytext=(0, 3),
                    textcoords='offset points', ha='center', fontsize=8, color=INK)
ax.set_xticks(x, [f'{s} ratings' for s in segs])
ax.set_ylabel('Share of users (%)')
ax.set_ylim(0, 65)
ax.set_title('About half of users have only 3-4 training ratings')
ax.legend(loc='upper right')
save(fig, 'fig3_train_segments')

In [ ]:
# ===== fig4. 상품 인기도 롱테일 =====
fig, ax = plt.subplots(figsize=(7, 4))
notes = []
for ds in DATASETS:
    cnt = data[ds]['train'].groupby('item_id').size().sort_values(ascending=False).values
    pct = np.arange(1, len(cnt) + 1) / len(cnt) * 100
    ax.plot(pct, cnt, color=COLOR[ds], linewidth=2, label=LABEL[ds])
    top10 = cnt[: max(1, len(cnt) // 10)].sum() / cnt.sum() * 100
    notes.append(f'{LABEL[ds].split(" (")[0]}: top 10% items = {top10:.0f}% of ratings')
ax.set_yscale('log')
ax.set_xlabel('Item popularity rank (percentile, most popular → least)')
ax.set_ylabel('Training ratings per item (log)')
ax.set_title('A few items receive most of the ratings')
ax.text(0.98, 0.95, '\n'.join(notes), transform=ax.transAxes, ha='right', va='top', fontsize=9, color=MUTED)
ax.legend(loc='lower left')
save(fig, 'fig4_item_long_tail')
print('\n'.join(notes))

In [ ]:
# ===== fig5. 연도 분포 (train vs test) =====
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, ds in zip(axes, DATASETS):
    for split in ['train', 'test']:
        yr = year_of(data[ds][split].timestamp)
        share = yr.value_counts(normalize=True).sort_index() * 100
        ax.plot(share.index, share.values, color=SPLIT_COLOR[split], linewidth=2, marker='o', markersize=4,
                label=f'{split} (median {int(yr.median())})')
        ax.annotate(split, (share.index[-1], share.values[-1]), xytext=(4, 0), textcoords='offset points',
                    color=INK, fontsize=9, va='center')
    ax.set_title(LABEL[ds], fontsize=11)
    ax.set_xlabel('Year')
    ax.legend(loc='upper left')
axes[0].set_ylabel('Share of ratings in split (%)')
fig.suptitle('Test ratings come 3-4 years after training ratings', fontweight='bold', color=INK, y=1.03)
save(fig, 'fig5_year_distribution')

## 다음

- `eda_summary.csv`의 `일치` 열이 모두 `O`인지 확인. `확인`이 있으면 반올림 차이인지 실제 차이인지 본다.
- 그림 5개는 Drive `results/eda/`에 있다. PPT 데이터 탐색 슬라이드에 그대로 쓴다.
- 다음 노트북: `03_sampling.ipynb` (프로토콜 ①②③ 후보 파일)